In [ ]:
import pandas as pd

# Load CSV file
df = pd.read_csv("pyannote_with_SN.csv")

# Define the RTTM file format
def convert_to_rttm(csv_file, rttm_file, file_id="test_audio"):
    df = pd.read_csv(csv_file)

    with open(rttm_file, "w") as f:
        for _, row in df.iterrows():
            start_time = row["start"]
            duration = row["end"] - row["start"]
            speaker = row["speaker"]
            f.write(f"SPEAKER {file_id} 1 {start_time:.3f} {duration:.3f} <NA> <NA> {speaker} <NA>\n")

# Convert CSV to RTTM
convert_to_rttm("pyannote_with_SN.csv", "pyannote_with_SN.rttm")

print("RTTM file created: pyannote_with_SN.rttm")


RTTM file created: pyannote_with_SN.rttm


In [ ]:
!pip install pyannote.metrics pyannote.core

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 3.1 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=97d8799dab18f331dce69e9ab49170a6ae45d40fb7e7791c37d7f7108c97fc4f
  Stored in directory: /root/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built docopt


In [ ]:
!pip install pyannote.core

In [ ]:
from pyannote.core import Annotation, Segment

# Load the RTTM file manually
def load_rttm_file(rttm_path, file_id="enhanced.wav"):
    with open(rttm_path, "r") as f:
        annotation = Annotation(uri=file_id)
        for line in f:
            parts = line.strip().split()
            start_time = float(parts[3])
            duration = float(parts[4])
            speaker = parts[7]
            annotation[Segment(start_time, start_time + duration)] = speaker
    return annotation

# Load ground truth and predicted RTTM files
reference = load_rttm_file("ground_truth.rttm")
hypothesis = load_rttm_file("pyannote_with_SN.rttm")

# Compute Diarization Error Rate (DER)
from pyannote.metrics.diarization import DiarizationErrorRate
der = DiarizationErrorRate()
der_value = der(reference, hypothesis)

print(f"Diarization Error Rate (DER): {der_value * 100:.2f}%")


Diarization Error Rate (DER): 74.91%


/usr/local/lib/python3.11/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
